# Caching and Performance Tuning
Companion notebook for the **Caching and Performance Tuning** section of Chapter 10.

Many requests repeat more than you'd expect, so caching at the right steps -- query, embedding, and result layers -- gives easy wins.

A minimal embedder and retriever, with a call counter so cache hits are visible:

In [1]:
call_counts = {"embed": 0, "search": 0}


class Embedder:
    def embed_query(self, text: str) -> list[float]:
        call_counts["embed"] += 1
        return [ord(ch) for ch in text[:8]]  # toy deterministic "embedding"


class Retriever:
    def search(self, vec: list[float], top_k: int = 20) -> list[dict]:
        call_counts["search"] += 1
        return [{"chunk_id": "c1", "score": sum(vec) % 100 / 100}]

This is the book's `timed_retrieve`, unchanged:

In [2]:
import time


def timed_retrieve(query, cache, embedder, retriever):
    t0 = time.perf_counter()
    q = query.strip().lower()

    if q in cache["results"]:
        return cache["results"][q], {"stage": "result_cache_hit"}

    if q in cache["embeddings"]:
        vec = cache["embeddings"][q]
    else:
        vec = embedder.embed_query(q)
        cache["embeddings"][q] = vec

    docs = retriever.search(vec, top_k=20)
    cache["results"][q] = docs
    return docs, {"latency_ms": (time.perf_counter() - t0) * 1000}

In [3]:
cache = {"results": {}, "embeddings": {}}
embedder, retriever = Embedder(), Retriever()

docs, meta = timed_retrieve("What is our refund window?", cache, embedder, retriever)
print("1st call (full miss):", meta, "| embed calls:", call_counts["embed"], "| search calls:", call_counts["search"])

docs, meta = timed_retrieve("What is our refund window?", cache, embedder, retriever)
print("2nd call (result cache hit):", meta, "| embed calls:", call_counts["embed"], "| search calls:", call_counts["search"])

cache["results"].pop("what is our refund window?")  # simulate the result cache expiring, but not the embedding cache
docs, meta = timed_retrieve("What is our refund window?", cache, embedder, retriever)
print("3rd call (embedding cache hit, result miss):", meta, "| embed calls:", call_counts["embed"], "| search calls:", call_counts["search"])

1st call (full miss): {'latency_ms': 0.005700043402612209} | embed calls: 1 | search calls: 1
2nd call (result cache hit): {'stage': 'result_cache_hit'} | embed calls: 1 | search calls: 1
3rd call (embedding cache hit, result miss): {'latency_ms': 0.002600019797682762} | embed calls: 1 | search calls: 2


Notice `embed` calls stay flat after the first call in every case above -- once cached, the same normalized query never pays the embedding cost again, even when the result cache alone is invalidated (call 3). Version the embedding cache key by model, so a model change can't silently serve stale vectors. Measure each stage separately (embed, retrieve, rerank, context build, generation) -- without stage timings, you optimize the wrong thing.